In [5]:
import hashlib
import math
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

# 1. 훈련 환경 설정
# 시드 고정하기
torch.manual_seed(2026)

# GPU 사용 여부 확인하기
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 한 번에 처리할 토큰의 최대 문맥 길이를 64로 설정하기
CONTEXT = 64 

# 2. 파일 불러 오기
path = Path("transcript.txt")
if not path.exists():
    # transcript.txt 없으면 에러
    raise FileNotFoundError("Place BeRP transcript.txt beside this notebook")


# 3. 텍스트 데이터를 읽어서 전치리하는 단계
records = []
for line in path.read_text(
    encoding="utf-8", 
    errors="replace"
).splitlines(): # 한 줄씩 읽기
    
    # 공백을 기준으로 분리하기
    parts = line.strip().split(maxsplit=1)
    # [ 데이터 ID, 실제 텍스트 ] -> 형태가 된다.    
    
    if len(parts) == 2 and parts[1].strip():
        records.append(
            (parts[0], parts[1].lower().split())
        ) # 실제 텍스트 문장을 단어 단위로 토큰화 해서 저장
        
    '''
    이런 형태가 됨!
    [
        ( 데이터 ID, [단어1, 단어2, 단어3, 단어4] ),
        ( 데이터 ID, [단어1, 단어2, 단어3, 단어4] ),
    ]
    '''

# 4. 학습 데이터와 검증 데이터를 분리하기
def bucket(record_id):
    return int.from_bytes(
        # ID를 해시 함수로 변환하고, 10으로 나눈 나머지를 사용
        hashlib.sha256(record_id.encode()).digest()[:8],
        "big"
    ) % 10

# 8 : 1 : 1
# 결과가 0과 7 사이에 있는 ID를 train set으로 사용
train = [
    words for uid, words in records 
    if bucket(uid) < 8
]

# 결과가 8인 ID를 validation set으로 사용
val = [
    (uid, words) for uid, words in records 
    if bucket(uid) == 8
]

# 검증 데이터 정렬하고 중복 제거하기 
val.sort(
    key=lambda row: hashlib.sha256(
        row[0].encode()
    ).digest()
)

val = [words for _, words in val]

# train과 validation 비어 있는 값을 확인
assert train and val, "Expected nonempty training and validation splits"

# Bucket 9 is reserved for the instructor's final evaluation.

# 5. Vocab 제작 
vocab = ["<pad>", "<bos>", "<eos>", "<unk>"]

vocab += sorted(
    set(w for sent in train for w in sent) - set(vocab)
)
'''
Vocab 제작 과정
1. train에서 모든 단어 추출 
2. set()으로 중복 제거
3. 기존 vocab과 차집합하여 기존에 없는 단어만 남김
4. sorted()로 정렬
5. vocab 단어장에 추가
'''

# 단어를 숫자로 변환하기 
stoi = {w: i for i, w in enumerate(vocab)}

# 특수 토큰에 숫자 ID를 저장하기
PAD, BOS, EOS, UNK = (
    stoi[w] for w in vocab[:4]
) 

# 단어 개수
VOCAB_SIZE = len(vocab)

# 6. Dataset 클래스 만들기
class LMData(Dataset):
    def __init__(self, sentences):
        self.pairs = []
        
        for words in sentences:
            # 단어를 숫자로 바꾸기
            ids = [BOS] + [
                stoi.get(w, UNK) for w in words
            ] + [EOS]
            
            # context만큼 잘라서 입려과 타겟 ( 입력에서 한 칸 뒤의 토큰들 ) 쌍을 생성
            for start in range(0, len(ids) - 1, CONTEXT):
                self.pairs.append((
                    torch.tensor(ids[start:start + CONTEXT]),
                    torch.tensor(ids[start + 1:start + CONTEXT + 1]),
                ))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.pairs[i]

# 7. Padding 처리하기 - 문장마다 길이가 다르기 때문에 길이를 동일하게 맞추기 ( 가장 긴 문장에 동일하게 맞춤 )
def collate(batch):
    return tuple(
        # pad_sequence()가 내부적으로 현재 배치에 있는 시퀀스들의 길이를 확인함!
        pad_sequence(
            parts,
            batch_first=True,
            padding_value=PAD
        )
        for parts in zip(*batch)
    )

# 8. DataLoader 만들기
train_loader = DataLoader(
    LMData(train), 
    batch_size=64, 
    shuffle=True, 
    collate_fn=collate
)

quick_loader = DataLoader(
    LMData(val[:256]),
    batch_size=64,
    collate_fn=collate
)

val_loader = DataLoader(
    LMData(val),
    batch_size=64,
    collate_fn=collate
)

# 9. 최종 결과 출력
print(
    f"device={DEVICE}, vocab={VOCAB_SIZE}, train={len(train)}, val={len(val)}"
)

device=cuda, vocab=1809, train=6829, val=904


In [6]:
# 교수님의 점수 채점 함수
def score(model, loader):
    # 파라미터 개수 카운트하는 부분 : 학습 가능한 것 만
    params = sum(
        p.numel() for p in model.parameters() 
        if p.requires_grad
    )
    # 200만 개 넘으면 채점 거부
    if not 0 < params <= 2_000_000:
        raise ValueError(f"Parameter budget exceeded: {params:,} (limit 2,000,000)")
    
    # 장치 확인
    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    nll, tokens = 0.0, 0
    try:
        with torch.inference_mode():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                if logits.shape != (x.size(0), x.size(1), VOCAB_SIZE):
                    raise ValueError("Expected logits (batch, time, VOCAB_SIZE)")
                nll += F.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE), y.reshape(-1),
                    ignore_index=PAD, reduction="sum",
                ).item()
                tokens += (y != PAD).sum().item()
    finally:
        model.train(was_training)
    ce = nll / tokens
    ppl = math.exp(ce)
    points = min(30.0, 600.0 / (ppl * (max(params, 100_000) / 1_000_000) ** 0.15))
    print(f"CE={ce:.3f} | PPL={ppl:.2f} | params={params:,} | score={points:.2f}/30")
    return {"ce": ce, "ppl": ppl, "params": params, "points": points}

In [7]:
# 과제 시작

import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. 설정 
# 훈련 단계에서 여기만 조정하면서 진행을 하기
HW6_CFG = {
    "vocab_size" : None, # 실제 vocab size를 적용하기 
    "context_length" : 64, # 위에서 CONTEXT 64
    "emb_dim" : 64,         # 토큰을 표현하는 임베딩 백터 차원
    "n_heads" : 4,          # MHA에서 Attention을 수행하는 Head의 개
    "n_layers" : 2,         # transformer block의 개 수 ( 조건 2개에서 4개 사이 )
    "drop_rate" : 0.1,      # DropOut 비율
    "qkv_bias" : False,     # Query, Key, Value에 Bias를 사용할지 여부
    "tie_weights" : True, # 입력층 W와 출력층 W를 공유하여 하나로 사용 - 밑바닥 LLM 책에서 GPT2 구현하는 과정에서 사용했음!
}

# 2. LayerNorm 층정규화 = 실제로는 nn.LayerNorm(emb_dim) 한 줄로 통일해서 사용가능
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5 # 앱실론 부여해서 모두 다 같은 수 일때 0으로 나누는 것을 방지
        self.scale = nn.Parameter(torch.ones(emb_dim)) 
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True) # 마지막 차원(특징) 평균
        var = x.var(dim=-1, keepdim=True, unbiased=False) # 분산 (n으로 나눔)
        norm_x = (x - mean) / torch.sqrt(var + self.eps) # 평균 0. 분산은 1로 정규화
        return self.scale * norm_x + self.shift
    
# 3. FF - FeedForward : d -> 4d -> d = nn.Sequential( nn.Linear(d, 4d), nn.GELU(), nn.Linear(4d, d) )
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # 넓히기
            nn.GELU(), # 비선형
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), # 다시 줄이기
        )

    def forward(self, x):
        return self.layers(x)

# 4. MultiHeadAttention = 실제로는 이렇게 간단히 사용가능 F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=...)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out은 num_heads로 나눠떨어져야 함"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 헤드 하나의 차원 (64/4 = 16)

        # Q, K, V를 만드는 선형층 3개 = nn.Linear(d_in, 3 * d_out) 하나로 합치고 split
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out) # 헤드들을 합친 뒤 섞는 층
        self.dropout = nn.Dropout(dropout)

        # causal mask : 대각선 위(미래)가 1인 상삼각 행렬
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape

        # (1) Q, K, V 만들기 : (b, T, d_out)
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # (2) 헤드로 쪼개기 : (b, T, d_out) -> (b, T, heads, head_dim) -> (b, heads, T, head_dim)
        # 하나의 큰 벡터를 여러 Head로 나눠서 각각 Attention을 수행, 여러 Head는 서로 다른 특징과 관계를 학습!@!@!
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        # (3) 어텐션 점수 - # Query와 Key의 내적을 통해 각 토큰 간의 연관성 점수 계산
        attn_scores = queries @ keys.transpose(2, 3)

        # (4) 미래 위치를 -inf로 가림 (softmax 후 0이 됨)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # (5) sqrt(head_dim)으로 나눠 스케일링 후 softmax -> 어텐션 가중치
        # 스케일링 이유 : 차원이 크면 점수가 커져 softmax가 한쪽으로 쏠림 -> 기울기 소실
        attn_weights = torch.softmax(attn_scores / self.head_dim ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # (6) Attention 가중합
        # Attention 가중치와 Value를 곱하여 중요한 정보들을 가중합
        # 각 Head별 최종 문맥 벡터(Context Vector) 생성
        context_vec = attn_weights @ values

        # (7) 헤드에서 나온 결과를 하나로 합치기 : (b, T, heads, head_dim) -> (b, T, d_out)
        # contiguous() : transpose로 흩어진 메모리를 연속으로 정리해야 view 가능
        context_vec = context_vec.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)


# 5. TransformerBlock - nn.TransformerEncoderLayer(d, nhead, 4*d, dropout, activation="gelu", batch_first=True, norm_first=True)
# + forward에 causal mask 전달
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # 어텐션 블록 + 숏컷(residual)
        shortcut = x
        x = self.norm1(x) # 층 정규화
        x = self.att(x) # 마스크드 멀티헤드 어텐션
        x = self.drop_shortcut(x) # 드롭아웃
        x = x + shortcut # 잔치연결(숏컷)

        # 피드포워드 블록 + 잔차연결(숏컷)
        shortcut = x
        x = self.norm2(x) # 층 정규화
        x = self.ff(x) # 피드 포워딩
        x = self.drop_shortcut(x) # 드롭아웃
        x = x + shortcut # 잔차 연결
        return x


# 6. GPTModel: 임베딩 -> 블록 x N -> 최종 LayerNorm -> 출력층
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) # 토큰 임베딩
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"]) # 위치 임베딩
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # 블록 N개 쌓기
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        # 최종 층 정규화
        self.final_norm = LayerNorm(cfg["emb_dim"])
        # 출력층
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

        # 출력층이 토큰 임베딩 행렬을 공유 ( 밑바닥 부터 배우는 LLM에 나온 방식)
        if cfg["tie_weights"]:
            self.out_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    # static void initWeight() static 메서드인 것!
    @staticmethod
    def _init_weights(m): # 모델의 가중치를 초기화
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, in_idx): # in_idx: (B, T)
        batch_size, seq_len = in_idx.shape
        
        tok_embeds = self.tok_emb(in_idx) # (B, T, D)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device)) # (T, D)
        
        x = tok_embeds + pos_embeds # 토큰 임베딩과 위치 임베딩을 더하기
        x = self.drop_emb(x) # 드롭아웃 - 과적합 방지
        x = self.trf_blocks(x) # 드랜스포머 블럭 여러개
        x = self.final_norm(x) # 최종 층 정규화
        logits = self.out_head(x) # (B, T, VOCAB_SIZE) <- score()가 요구하는 모양
        return logits

In [8]:
cfg = dict(HW6_CFG, vocab_size=VOCAB_SIZE, context_length=CONTEXT)
model = GPTModel(cfg).to(DEVICE)

x, y = next(iter(quick_loader))
print(model(x.to(DEVICE)).shape)

for _ld in (train_loader, quick_loader, val_loader):
    _ld.dataset.pairs = [(x[:len(y)], y) for x, y in _ld.dataset.pairs]
    
# 학습 전 점수를 확인하기
score(model, quick_loader)

torch.Size([64, 19, 1809])
CE=7.506 | PPL=1819.22 | params=219,584 | score=0.41/30


{'ce': 7.506163989681754,
 'ppl': 1819.2215880023189,
 'params': 219584,
 'points': 0.414024027374812}